In [1]:
import pandas as pd


In [5]:
# Load the three tables
raw_edges = pd.read_csv("raw_edges.csv")
edges = pd.read_csv("edges.csv")
corrections = pd.read_csv("corrections.csv")

output = "confidence_rank_raw.csv"

In [3]:
# Keys identifying an edge
keys = ["Regulator", "Target", "Sign", "Model"]


def unique_rows(df1, df2):
    """Rows whose keys occur in df1 but not in df2."""
    return (
        df1.merge(
            df2[keys].drop_duplicates(),
            on=keys,
            how="left",
            indicator=True,
        )
        .query('_merge == "left_only"')
        .drop(columns="_merge")
    )

In [4]:
# ------------------------------------------------------------
# Report unique rows
# ------------------------------------------------------------

raw_vs_edges = unique_rows(raw_edges, edges)
edges_vs_raw = unique_rows(edges, raw_edges)

raw_vs_corrections = unique_rows(raw_edges, corrections)
correction_vs_raw = unique_rows(corrections, raw_edges)

# Overlap between the two new CSVs
overlap = edges.merge(
    corrections[keys].drop_duplicates(),
    on=keys,
    how="inner",
).drop_duplicates(subset=keys)


print("=== Unique rows ===")
print(f"Unique to raw_edges vs edges : {len(raw_vs_edges)}")
print(f"Unique to edges vs raw_edges : {len(edges_vs_raw)}")
print(f"Unique to raw_edges vs corrections : {len(raw_vs_corrections)}")
print(f"Unique to corrections vs raw_edges : {len(correction_vs_raw)}")

print("\n=== Overlap between the two new CSVs ===")
print(f"Overlapping edges: {len(overlap)}")

=== Unique rows ===
Unique to raw_edges vs edges : 0
Unique to edges vs raw_edges : 0
Unique to raw_edges vs corrections : 1587
Unique to corrections vs raw_edges : 80

=== Overlap between the two new CSVs ===
Overlapping edges: 0


In [6]:
# ------------------------------------------------------------
# Merge both new CSVs onto raw_edges
# ------------------------------------------------------------

result = raw_edges.copy()

for df in [edges, corrections]:

    # Columns that exist only in the new dataframe
    extra_columns = [c for c in df.columns if c not in result.columns]

    print(f"\nAdding {len(df)} rows/columns")
    print("Extra columns:", extra_columns)

    # Transfer extra columns for existing edges
    if extra_columns:
        result = result.merge(
            df[keys + extra_columns],
            on=keys,
            how="left",
        )

    # Find genuinely new edges
    new_rows = unique_rows(df, result)

    # Match result's column structure
    for col in result.columns:
        if col not in new_rows.columns:
            new_rows[col] = pd.NA

    new_rows = new_rows[result.columns]

    # Append new edges
    result = pd.concat([result, new_rows], ignore_index=True)

    print(f"Rows added: {len(new_rows)}")


print("\n=== Final ===")
print(f"Original raw_edges: {len(raw_edges)} rows")
print(f"Final table:       {len(result)} rows")

result.to_csv(output, index=False)

display(result.head())


Adding 1587 rows/columns
Extra columns: ['Classification', 'Confidence rank', 'Constraint', 'Notes']
Rows added: 0

Adding 80 rows/columns
Extra columns: []
Rows added: 80

=== Final ===
Original raw_edges: 1587 rows
Final table:       1667 rows


,index,Regulator,Target,Sign,Model,Classification,Confidence rank,Constraint,Notes
0,1,AKT,AKT,positive,BL,indirect,2,NaN,AKT does not self-regulate but it participates...
1,2,EGFR,AKT,positive,BL,indirect,2,NaN,"causal effect mediated by PI3K, others"
2,3,HER2,AKT,positive,BL,indirect,2,NaN,"causal effect mediated by PI3K, others"
3,4,HER3,AKT,positive,BL,indirect,2,NaN,"causal effect mediated by PI3K, others"
4,5,PTEN,AKT,negative,BL,indirect,2,NaN,causal effect mediated by PIP3
